# Installation & Setup

**`pip install pytest-playwright`**

* The `pytest-playwright` plugin provides a lot of ready-made, useful fixtures such as fixtures for launching a new page and handling the closing of the playwright session, etc.
* With the `pytest-playwright` plugin, we just need to focus on writing tests. For any boilerplate code/common functionality, `pytest-playwright` will have a fixture defined already.
                                                                                                                                                   

# Writing test with `pytest-playwright` plugin

**Example**: without the `pytest-playwright` plugin, we need to explicitly handle launching the browser & closing the playwright session.

```python
import pytest
from playwright.sync_api import sync_playwright

@pytest.fixture
def page():
    playwright = sync_playwright().start()
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    return browser.new_page()

# page fixture
def test_page_has_get_started_link(page):
    page.goto("https://playwright.dev/python")

    link = page.get_by_role("link", name="GET STARTED")
    link.click()
    assert page.url == "https://playwright.dev/python/docs/intro"
```

**Example**: with the `pytest-playwright` plugin, we get **out-of-the box** standard fixture called **Page** which returns a **Page** object, handles launching the browser & closing the playwright session.

```python
from playwright.sync_api import Page

def test_page_has_get_started_link(page: Page):
    page.goto("https://playwright.dev/python")

    link = page.get_by_role("link", name="GET STARTED")
    link.click()
    assert page.url == "https://playwright.dev/python/docs/intro"

```

# Running tests

* **Run all tests in headless mode**: `pytest`
* **Run single test file in headless mode**: `pytest demo01_pytest_playwright_test.py`
* **Run in headed mode**: `pytest --headed --slowmo=1000 demo01_pytest_playwright_test.py`
* **Run in a different browser other than the default Chrome**: `pytest --headed --slowmo=1000 --browser=firefox demo01_pytest_playwright_test.py`

# Pytest Config file

* To run tests, we can simply use the `pytest` commands & options.
* Sometimes, the number of command-line options increases to an extent that it becomes tedious to provide all those options manually everytime while executing tests.
* A better option is to use the pytest config file `pytest.ini`.
* We can simply define the configuration in the `pytest.ini` file and place it in the project root directory.
* Then, every time we run the `pytest` command, pytest will fetch the commandline options and configuration from the `pytest.ini` file and exute accordingly.

**`pytest.ini`**

```
[pytest]
addopts = --headed --slowmo=500 --browser=firefox
```

# Test Hooks

**Hooks** are fixture that runs before & after tests automatically without passing the fixture name to the test function.

```python

import pytest
from playwright.sync_api import Page

DOCS_URL = "https://playwright.dev/python/docs/intro"


# HOOKS
# The "autouse=True" argument runs the fixture before each test function automatically without specifying it in the test method
@pytest.fixture(autouse=True, scope="function")
def visit_palywright_website(page: Page):
    # setup steps
    page.goto("https://playwright.dev/python")
    # teardown steps
    # code executes after each test function using yield keyword, which makes it a generator function
    yield page
    page.close()

# Note that we're not passing the fixture name as argument to the test function
def test_page_has_doc_link(page: Page):
    link = page.get_by_role("link", name="Docs")
    assert link.is_visible()

# Note that we're not passing the fixture name as argument to the test function
def test_get_started_visits_docs(page: Page):
    link = page.get_by_role("link", name="GET STARTED")
    link.click()
    assert page.url == DOCS_URL

```

# Function Scope Fixtures

Up until now, we've used the `Page` fixture inside our tests, which is a **function-scoped** fixture.

A **function-scoped** fixture is provided to each test function, which provides a new object as per the fixture type. For example, the `Page` fixture will provide a **Page** object to the test function.

Similarly, we've another function-scoped fixture called `BrowserContext`

**Example**:

```python
from playwright.sync_api import Page, BrowserContext


def test_page_has_doc_link(context: BrowserContext):
    # context.add_cookies()
    # context.grant_permissions()
    
    page = context.new_page()
    page.goto("from playwright.sync_api import Page")
    link = page.get_by_role("link", name="Docs")
    link.highlight()
    assert link.is_visible()

```

# Session Scope Fixtures

The **Session scope** fixtures are only created once per test session and provided to all of the test functions. For example. `Browser` fixture, which is launched once and provided to all test functions.

Some other **Session scope** fixtures are: `Playwright`, `BrowserType`, etc.

```python

import pytest
from playwright.sync_api import Browser

DOCS_URL = "https://playwright.dev/python/docs/intro"

# Passing Browser fixture
def test_page_has_doc_link(
    playwright: Playwright,
    browser: Browser,
    browser_type: BrowserType,
    browser_name: str,
    browser_channel: str,
    is_firefox: bool,
    is_chromium: bool,
    is_webkit: bool,
): 
    page = browser.new_page()

    if is_firefox:
        pass # firefox specific code

    if browser_type == playwright.chromium:
        pass # chromium specific code
    elif browser_type == playwright.firefox:
        pass # firefox specific code
    
    page.goto("https://playwright.dev/python")
    link = page.get_by_role("link", name="Docs")
    assert link.is_visible()


# This will be the same Browser fixture as passed in the previous test, as the Browser fixture is a Session scope fixture.
# Browser instance will only be created once per test session and will be used in all test functions for that session.
def test_get_started_visits_docs(browser: Browser): 
    page = browser.new_page()
    page.goto("https://playwright.dev/python")
    link = page.get_by_role("link", name="GET STARTED")
    link.click()
    assert page.url == DOCS_URL

```



# Browser Selection Markers


```python

import pytest
from playwright.sync_api import Page

DOCS_URL = "https://playwright.dev/python/docs/intro"

@pytest.mark.skip_browser("firefox")          # skip execution of this test in firefox browser
def test_page_has_doc_link(page: Page):
    page.goto("https://playwright.dev/python")
    link = page.get_by_role("link", name="Docs")
    assert link.is_visible()


@pytest.mark.only_browser("firefox")          # executes this test only if the browser is a firefox browser
def test_get_started_visits_docs(page: Page):
    page.goto("https://playwright.dev/python")
    link = page.get_by_role("link", name="GET STARTED")
    link.click()
    assert page.url == DOCS_URL

```



# Browser Launch And Context Arguments

So far, the fixture we've used we doesn't get to configure the browser launch options. That is, we cannot define the arguments to configure the browser launch session. We'll now take a look at how to pass the arguments when we launch our browser and create a new context.

**Example**:

```python

import pytest
from playwright.sync_api import Page

@pytest.fixture(scope="session")
def browser_type_launch_args():
    return {
        **browser_type_launch_args,
        "headless": False,
        "slow_mo": 1000
    }


@pytest.fixture(scope="session")
def browser_context_arg(browser_context_args):
    return {
        **browser_context_args,
        "storage_state": "playwright/.auth/storage_state.json"
    }

def test_page_has_doc_link(page: Page):
    page.goto("https://playwright.dev/python")
    link = page.get_by_role("link", name="Docs")
    assert link.is_visible()
    

```